# Exploring the data
This is the notebook where I will explore the data and see what columns are in the dataset.
The training data lies in *../data/train.En.csv* and the two test datasets lies in *../data/task_A_En_test.csv* and *../data/task_B_En_test.csv*.

In [107]:
#imports
import pandas as pd

## Loading the data

In [124]:
train_data = pd.read_csv("../data/train.En.csv")

## Columns and size

In [125]:
train_columns = train_data.columns
print(f"Columns in the training dataset: {train_columns}")

test_data_A = pd.read_csv("../data/task_A_En_test.csv")
test_data_B = pd.read_csv("../data/task_B_En_test.csv")
test_A_columns = test_data_A.columns
test_B_columns = test_data_B.columns
print(f"Columns in the test dataset A: {test_A_columns}")
print(f"Columns in the test dataset B: {test_B_columns}")

Columns in the training dataset: Index(['Unnamed: 0', 'tweet', 'sarcastic', 'rephrase', 'sarcasm', 'irony',
       'satire', 'understatement', 'overstatement', 'rhetorical_question'],
      dtype='str')
Columns in the test dataset A: Index(['text', 'sarcastic'], dtype='str')
Columns in the test dataset B: Index(['text', 'sarcasm', 'irony', 'satire', 'understatement', 'overstatement',
       'rhetorical_question'],
      dtype='str')


Found that the columns in the dataset consists of one field that is unnamed (the index), then the tweet itself. After this follows the "sarcastic" field that tells us if the tweet is sarcastic or non-sarcastic. Next is the rephrases that is the "translation" of the sarcasm. And after that is 6 different sarcasm categories that we need for task B. 

In [109]:
train_rows = train_data.shape
print(f"Number of (rows, columns) in train file: {train_rows}")

test_A_rows = test_data_A.shape
test_B_rows = test_data_B.shape
print(f"Number of (rows, columns) in test file A: {test_A_rows}")
print(f"Number of (rows, columns) in test file B: {test_B_rows}")

Number of (rows, columns) in train file: (3468, 10)
Number of (rows, columns) in test file A: (1400, 2)
Number of (rows, columns) in test file B: (1400, 7)


This means that there are 3468 tweets to train and validate on. 

## Class distribution

In [110]:
train_number_of_sarcastic = train_data.value_counts("sarcastic")
print(f"Number of sarcastic tweets in train set: {train_number_of_sarcastic}")
train_number_of_sarcastic_share = train_data.value_counts("sarcastic", normalize=True)
print(f"Share of sarcastic tweets in train set: {train_number_of_sarcastic_share}")

test_A_sarcastic = test_data_A.value_counts("sarcastic")
print(f"Number of sarcastic tweets in test set A: {test_A_sarcastic}")
test_A_sarcastic_share = test_data_A.value_counts("sarcastic", normalize=True)
print(f"Share of sarcastic tweets in test set A: {test_A_sarcastic_share}")

Number of sarcastic tweets in train set: sarcastic
0    2601
1     867
Name: count, dtype: int64
Share of sarcastic tweets in train set: sarcastic
0    0.75
1    0.25
Name: proportion, dtype: float64
Number of sarcastic tweets in test set A: sarcastic
0    1200
1     200
Name: count, dtype: int64
Share of sarcastic tweets in test set A: sarcastic
0    0.857143
1    0.142857
Name: proportion, dtype: float64


We can see that 867/3468 tweets in the train dataset are sarcastic and 2601 are non-sarcastic. That means that 1/4 of the dataset is sarcastic and the rest is not. 

And in the A test set 200/1400 tweets are sarcastic, about 14.3%, which is lower than the training set (!).

Test set B does not have a "sarcastic" column, but contains all the same 1400 tweets as test set A.

## Missing values and duplicates

In [111]:
missing_values = train_data.isna().sum()
print(missing_values)

Unnamed: 0                0
tweet                     1
sarcastic                 0
rephrase               2601
sarcasm                2601
irony                  2601
satire                 2601
understatement         2601
overstatement          2601
rhetorical_question    2601
dtype: int64


One tweet is missing the text. And the 2601 missing values in the rephrase and categories are the non-sarcastic tweets.

In [112]:
missing_tweet = train_data[train_data["tweet"].isna()]
print(missing_tweet)

      Unnamed: 0 tweet  sarcastic rephrase  sarcasm  irony  satire  \
1062        1062   NaN          0      NaN      NaN    NaN     NaN   

      understatement  overstatement  rhetorical_question  
1062             NaN            NaN                  NaN  


The missing tweet has the index 1062 and this needs to be removed from the data that will be used for the training. The tweet is marked as non-sarcastic, therefore the removal only affects the larger class.

In [113]:
missing_test_A = test_data_A.isna().sum()
print(missing_test_A)

text         0
sarcastic    0
dtype: int64


No missing tweets in the A test.

In [114]:
duplicates = train_data.duplicated(subset="tweet")
# print(duplicates)
print(f"Number of duplicates: {duplicates.sum()}")

Number of duplicates: 11


In [115]:
duplicate_rows = train_data[train_data["tweet"].duplicated(keep=False)]
# print(duplicate_rows)
print(duplicate_rows.sort_values("tweet")[["tweet", "sarcastic"]])

                                                  tweet  sarcastic
47    @AlexShawESPN @robwishart @GNev2 Does he order...          1
1008  @AlexShawESPN @robwishart @GNev2 Does he order...          0
24    @AsdaServiceTeam imagine your delivery being 2...          1
939   @AsdaServiceTeam imagine your delivery being 2...          0
940   @AsdaServiceTeam imagine your delivery being 2...          0
941   @AsdaServiceTeam imagine your delivery being 2...          0
927                                   @kevinabstract ye          0
928                                   @kevinabstract ye          0
2070  Great effort @Patriots We'll get em all next year          0
2071  Great effort @Patriots We'll get em all next year          0
3067  Omg flashbacks to emma blackery girl balls mer...          0
3068  Omg flashbacks to emma blackery girl balls mer...          0
338                               Replace Pelosi #Nancy          1
1881                              Replace Pelosi #Nancy       

In [116]:
labels_per_tweet = duplicate_rows.groupby("tweet")["sarcastic"].nunique()
print(labels_per_tweet)
print(f"Number of duplicated tweets: {len(labels_per_tweet)}")
print(f"Tweets with conflicting labels: {(labels_per_tweet > 1).sum()}")

tweet
@AlexShawESPN @robwishart @GNev2 Does he order 11 at once and drink each one at just over 5 minutes, OR do we have to take into account going to the bar? Either way it’s a disgrace and not what we should be promoting to the kids.                  2
@AsdaServiceTeam imagine your delivery being 2 hours late, and imagine calling up your service team only for them to hang up at 10pm, coincidentally the same time the office closes. But it’s okay, my £3 delivery fee is being refunded though 👊🏻    2
@kevinabstract ye                                                                                                                                                                                                                                      1
Great effort @Patriots We'll get em all next year                                                                                                                                                                                                      

I found 11 duplicates of 9 tweets. 5 of them have conflicting labels for the sarcastic column.

In [117]:
duplicate_test_A = test_data_A["text"].duplicated()
print(f"Number of duplicates in test set A: {duplicate_test_A.sum()}")

Number of duplicates in test set A: 6


In [118]:
duplicate_rows_test = test_data_A[test_data_A["text"].duplicated(keep=False)]
# print(duplicate_rows)
print(duplicate_rows_test.sort_values("text")[["text", "sarcastic"]])

                                                   text  sarcastic
162   I see all the reasons Mark has put forward and...          0
647   I see all the reasons Mark has put forward and...          0
136   I'm a bit biased but I have to say this was fa...          0
326   I'm a bit biased but I have to say this was fa...          0
268   Im just gonna go out there and say it… Chas an...          0
786   Im just gonna go out there and say it… Chas an...          0
380        Norwich need to score whilst they’re on top.          0
625        Norwich need to score whilst they’re on top.          0
877                 One of my favorite stops in London!          0
903                 One of my favorite stops in London!          0
855                       So stoked to see them tonight          0
1068                      So stoked to see them tonight          0


In [119]:
labels_per_tweet_test = duplicate_rows_test.groupby("text")["sarcastic"].nunique()
print(labels_per_tweet_test)
print(f"Number of duplicated tweets: {len(labels_per_tweet_test)}")
print(f"Tweets with conflicting labels: {(labels_per_tweet_test > 1).sum()}")

text
I see all the reasons Mark has put forward and they are well explained but you need to ask yourself what is he looking at getting out of this ??                                                    1
I'm a bit biased but I have to say this was fantastic. The talent in the young people on our island is amazing. Well done everyone.\nPlease retweet if you saw the production and agree with me.    1
Im just gonna go out there and say it… Chas and Dave are underrated in my opinion.                                                                                                                  1
Norwich need to score whilst they’re on top.                                                                                                                                                        1
One of my favorite stops in London!                                                                                                                                                                 1
So st

In test set A I found 6 duplicates, where 0 of them had conflicting labels.

## Example tweets

10 sarcastic tweets (examples):

In [120]:
sarcastic_tweets = train_data[train_data["sarcastic"] == 1].sample(10, random_state=42)
# print(sarcastic_tweets)
for tweet in sarcastic_tweets["tweet"]:
    print("-", tweet)

- All the shade i have been hearing about Ben Platt being unbelievable as a teenager in the @DearEvanHansen movie  boggles me.  Does nobody have any memories of Grease???  Name 1 actor on that film who believably looked high school age!
- you would think the odds of me sitting next to the exact same stinky man three bus trips in a row is highly unlikely. alas I seem to be a lucky gal
- Billy Gunn sorta relevant for the first time in years...
- I see Dettol, Toilet Duck and Zoflora are all trending on Twitter. Now why could that be... #TrumpPressBriefing
- @catboychika Aren't you happy I gave you less competition for ranking
- Mad how many cars they make now without indicators 🙃
- oh also, my song with my middle school boyfriend was “143” by Bobby Brackins!! i really peaked geez
- I really hate the new TikTok voice over voice - she sound annoying af
- Oh my goodness. It’s the first week of the summer holidays and Harrison has found his recorder. 

Give.
Me.
Strength.
- Agreeing with the

10 non-sarcastic tweets (examples):

In [126]:
non_sarcastic_tweets = train_data[train_data["sarcastic"] == 0].sample(10, random_state=42)
# print(non_sarcastic_tweets)
for tweet in non_sarcastic_tweets["tweet"]:
    print("-", tweet)

- wild that theres a whole industry of pop ups that are like just elaborate ig photo ops and people are paying for them???
- Vaccine dose 1. Thank you, science.
- Sad because I’ve missed so much of Oxford the past year but happy cuz I just saw a squirrel carrying it’s baby whilst writing my essay by the river I love it https://t.co/A2AlOimCLs
- women say they want a man who has his life together. so how come whenever i show one the seven days’ worth of microwaved broccoli, carrot circles and chicken breast i keep in a tupperware in my fridge at all times, they are instantly rendered catatonic with clinical depression
- im just tryna get in n out doordashed to india rn
- I wish i loved my body
- I'm a little late but the Hamilton soundtrack is actually life-changing
- got a google calendar notif, my boyfriend put "think about dwarves" in as an all day event
- Passed my theory
- I work in DC. I live 15 minutes from the Capitol Building. I walk my dog down the National Mall at least once 

## Tweet length

In [122]:
train_data["n_words"] = train_data["tweet"].str.split().str.len()
# print(train_data.groupby("sarcastic")["n_words"])
# print(train_data.groupby("sarcastic")["n_words"].mean())
print(train_data.groupby("sarcastic")["n_words"].describe())

test_data_A["n_words"] = test_data_A["text"].str.split().str.len()
print(test_data_A.groupby("sarcastic")["n_words"].describe())

            count       mean        std  min   25%   50%   75%   max
sarcastic                                                           
0          2600.0  19.012308  11.597979  1.0  10.0  17.0  25.0  61.0
1           867.0  18.296424  10.241361  1.0  11.0  16.0  24.0  57.0
            count       mean        std  min   25%   50%    75%    max
sarcastic                                                             
0          1200.0  15.420833   8.864774  1.0   9.0  14.0  19.00  128.0
1           200.0  22.275000  12.719209  4.0  14.0  20.0  27.25  109.0


We can see that the max length for the test set is much longer than for the training set. The mean length for sarcastic and non-sarcastic tweets in the training set is a bit similar. But in test set A, the sarcastic tweets are clearly a bit longer in general. The difference means that the test and training data is not similar, and the model could not have learned to use length as a sign of sarcasm since the length is about the same for the sarcastic and non-sarcastic tweets in the training set.

## Sarcasm categories

In [123]:
categories = ["sarcasm", "irony", "satire", "understatement", "overstatement", "rhetorical_question"]
print("Train set:")
print(train_data[categories].sum())

print("Test set B:")
print(test_data_B[categories].sum())

sarcastic_train = train_data[train_data["sarcastic"] == 1]
print("Number of categories per sarcastic tweet (train):")
print(sarcastic_train[categories].sum(axis=1).value_counts())

categories_per_tweet_test = test_data_B[categories].sum(axis=1)
print("Number of categories per tweet (test B):")
print(categories_per_tweet_test.value_counts())

Train set:
sarcasm                713.0
irony                  155.0
satire                  25.0
understatement          10.0
overstatement           40.0
rhetorical_question    101.0
dtype: float64
Test set B:
sarcasm                180
irony                   20
satire                  49
understatement           1
overstatement           10
rhetorical_question     11
dtype: int64
Number of categories per sarcastic tweet (train):
1.0    692
2.0    173
3.0      2
Name: count, dtype: int64
Number of categories per tweet (test B):
0    1200
1     130
2      69
3       1
Name: count, dtype: int64


We see that "sarcasm" is the most common category in both sets. One tweet can have multiple categories, so the sum of categories is larger than the number of sarcastic tweets. The category "understatement" only has one tweet in the test set, which makes it hard to draw conclusions about that category in the analysis. The category "satire" is represented a lot more in the test set, even though there are fewer sarcastic tweets in total.

## Summary

### Size and class distribution
- The training set contains 3468 tweets, of which 867 (25%) are sarcastic.
- Test set A contains 1400 tweets, of which 200 (14.3%) are sarcastic. The share of sarcasm is lower in the test set than in the training set. The models will be trained on a higher share of sarcastic tweets than they are tested on, which may lead them to predict "sarcastic" too often and lower the precision.
- Test set B contains the same 1400 tweets, with category labels instead of the "sarcastic" column.

### Data cleaning (training set only)
- One tweet is missing its text and will be removed.
- There are 9 tweets that appear more than once (11 extra copies). For the 4 tweets with the same label on every copy, one copy will be kept. The 5 tweets with conflicting labels will be removed completely, since the correct label is unknown.
- This will leave 3451 tweets, where 862 are sarcastic. 
- The test set will not be changed. It will include its 6 duplicates so that the results can be compared with the other existing results of the task.

### Tweet length
- In the training set, sarcastic and non-sarcastic tweets have about the same length (18.3 and 19.0 words on average).
- In test set A, the sarcastic tweets are clearly longer (22.3 vs. 15.4 words), which is a difference between the training and test data.
- The longest tweet in the test set has 128 words, so a few tweets may be cut off by the transformers' maximum input length (?).

### Sarcasm categories
- "sarcasm" is by far the most common category in both sets.
- A tweet can belong to more than one category.
- Some categories are very small, especially "understatement", which has only one tweet in the test set. It can be hard to draw conclusions about these categories in the error analysis.
- "satire" is much more common in the test set (49) than in the training set (25), even though the test set has fewer sarcastic tweets.

### Example tweets
- Some of them includes "positive words about something negative".
- Some have rhetorical questions.
- Some includes emojis that often is used to mark irony.
- Some of the sarcastic tweets cannot be understood without knowing things about the world.
- Some sarcastic tweets don't look sarcastic.
- Some non-sarcastic tweets could be read as sarcastic.
- Both sarcastic and non-sarcastic tweets contains not formal language and slang, so that cannot be used to decide.
